In [1]:
import os
import numpy as np
from scipy import stats
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [7]:
def outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xz[xz > z] = np.nan # remove values above 3 z score
    xz[xz < -z] = np.nan # remove values below -3 z score
    xz = xz.fillna(method='ffill') # change nans to previous value
    xz = xz.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xz

def emp_outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xc[xz > z] = np.nan # remove values above 3 z score
    xc[xz < -z] = np.nan # remove values below -3 z score
    xc = xc.fillna(method='ffill') # change nans to previous value
    xc = xc.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xc

def corrmean(data):
    return np.tanh(np.nanmean(np.arctanh(data)))

In [5]:
dataDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/experiment 2/'
os.chdir(dataDir)

context_rating_data = {}
context_gaze_data = {}

# iterate through folders
## gaze data
for folder in os.listdir():
    
    os.chdir(dataDir + folder)

    # identify and store core files
    gazeFile = [x for x in os.listdir() if 'gaze' in x][0]
    ratingFile = folder + '.csv'

    ## gaze data
    df = outlier_remover(pd.read_csv(gazeFile).iloc[:,1:].dropna())

    # create new dataframe for column
    for column in df.columns:
            
        # store data

        if column not in context_gaze_data:
            context_gaze_data[column] = pd.DataFrame()
        context_gaze_data[column] = pd.concat([context_gaze_data[column],df[column].rename(folder)], axis = 1)

    ## rating data
    df = outlier_remover(pd.read_csv(ratingFile).iloc[:,1:].dropna())

    # create new dataframe for column
    for column in df.columns:
        if column not in context_rating_data:
            context_rating_data[column] = pd.DataFrame()

        # store data
        if column not in context_rating_data:
            context_rating_data[column] = pd.DataFrame()

        context_rating_data[column] = pd.concat([context_rating_data[column],df[column].rename(folder)], axis = 1)
        
    os.chdir(dataDir)


In [8]:
## context condition
context_taskPerf = {}
context_taskPerf_valence = {}
context_taskPerf_arousal = {}

# iterate through the videos
for video in context_rating_data:
    if 'ground' in video:
        continue
    
    df = context_rating_data[video].dropna() # drop na rows

    for subj in df.columns:
        if subj not in context_taskPerf: # add subject to our dict
            context_taskPerf[subj] = []
            context_taskPerf_valence[subj] = []
            context_taskPerf_arousal[subj] = []

        sr = df[subj]
        cr = df.drop(columns = [subj]).mean(axis = 1) # traditional consensus
        # cr = compute_ewe(df.drop(columns = [subj])) # ewe consensus
        # cr = compute_cct(df.drop(columns = [subj]))
        
        r, p = stats.spearmanr(sr,cr)
        context_taskPerf[subj].append(r)

        if 'valence' in video:
            context_taskPerf_valence[subj].append(r)
        else:
            context_taskPerf_arousal[subj].append(r)

# compute average performance
context_avg_taskPerf = {s:corrmean(context_taskPerf[s]) for s in context_taskPerf}
context_avg_taskPerf_valence = {s:corrmean(context_taskPerf_valence[s]) for s in context_taskPerf_valence}
context_avg_taskPerf_arousal = {s:corrmean(context_taskPerf_arousal[s]) for s in context_taskPerf_arousal}


In [34]:
import pandas as pd

window = 150
# Initialize result dictionary, filtering out 'ground' videos
disc_videos = {v: pd.DataFrame() for v in context_rating_data if 'ground' not in v}

for video_name, df_raw in context_rating_data.items():
    
    # Clean, check, and calculate essential variables
    df = df_raw.dropna(axis=0, how='any') 
    num_subjects = len(df.columns)
    n_minus_1 = num_subjects - 1
    
    # Vectorized LOO score calculation (single line)
    df_loo = (df.sum(axis=1).values[:, None] - df) / n_minus_1
    
    # Calculate and store rolling correlation in a dictionary comprehension (single line)
    disc_videos[video_name] = pd.DataFrame({
        subj: df_loo[subj].rolling(window).corr(df[subj])
        for subj in df.columns
    })

In [51]:
import pandas as pd
import numpy as np

def align_and_restructure_data(dict1, dict2):
    """
    Combines valence/arousal (ratings, dict1) and gaze coordinates (dict2) by 
    aligning the higher-frequency data (assumed to be dict2) to the lower-
    frequency data (assumed to be dict1) using linear interpolation.
    
    The final structure is {video_id: {frame_number: DataFrame}}.
    """
    
    frame_indexed_data = {}
    video_ids = sorted(list(set(k.split('_')[0] for k in dict1.keys())))
    
    for video_id in video_ids:
        try:
            # 1. Load all four dataframes
            df_valence = dict1[f'{video_id}_valence']
            df_arousal = dict1[f'{video_id}_arousal']
            df_xcoord = dict2[f'{video_id}_gazeX']
            df_ycoord = dict2[f'{video_id}_gazeY']
        except KeyError as e:
            print(f"Skipping {video_id}: Missing expected key {e}")
            continue

        # --- 2. Determine Target Length and Align Data ---
        
        # Assume ratings (dict1) have the lower count (target length)
        target_len = len(df_valence)
        
        # The coordinates (dict2) DataFrames are the source that may need resampling
        df_coords = {'gazeX': df_xcoord, 'gazeY': df_ycoord}
        
        # New dictionary to hold all aligned/resampled DataFrames
        aligned_frames = {
            'valence': df_valence,
            'arousal': df_arousal
        }
        
        # Iterate over the coordinate DataFrames
        for dim, df_source in df_coords.items():
            source_len = len(df_source)
            
            if source_len != target_len:
                print(f"Aligning {dim} data for {video_id}: {source_len} -> {target_len}")
                
                # Resample the source data to match the target length
                
                # Create a new index representing the frame numbers of the target length
                new_index = np.linspace(0, source_len - 1, target_len)
                
                # Create a temporary DataFrame with a float index representing the 
                # true position of the original high-frequency data.
                df_temp = df_source.copy()
                df_temp.index = np.arange(source_len)
                
                # Reindex to the new, non-integer index. This introduces NaNs at 
                # all the interpolated points.
                df_temp = df_temp.reindex(df_temp.index.union(new_index)).sort_index()
                
                # Interpolate the NaNs using linear interpolation
                df_resampled = df_temp.interpolate(method='linear').loc[new_index]
                
                # Reset the index to be simple integers (0 to target_len - 1)
                df_resampled.index = np.arange(target_len)
                
                aligned_frames[dim] = df_resampled
            else:
                aligned_frames[dim] = df_source.copy()
        
        # --- 3. Combine and Restructure to Frame Dictionary (as before) ---
        
        # Concatenate all four aligned DataFrames
        combined_df = pd.concat(
            list(aligned_frames.values()),
            axis=1,
            keys=list(aligned_frames.keys())
        )
        combined_df.columns.names = ['Dimension', 'Participant_ID']

        # Restructure using dictionary comprehension for final output
        frame_dict = {
            frame_num: combined_df.loc[frame_num].unstack(level='Dimension').reset_index().rename_axis('Participant_ID')
            for frame_num in combined_df.index
        }
        
        frame_indexed_data[video_id] = frame_dict
        
    return frame_indexed_data

In [52]:
frame_indexed_data = align_and_restructure_data(disc_videos, context_gaze_data)
#
# Accessing the data:
frame_indexed_data['011'][100] # DataFrame for video 011, frame 45

Aligning gazeX data for 004: 2984 -> 1692
Aligning gazeY data for 004: 2984 -> 1692
Aligning gazeX data for 005: 2984 -> 1692
Aligning gazeY data for 005: 2984 -> 1692
Aligning gazeX data for 011: 2984 -> 1692
Aligning gazeY data for 011: 2984 -> 1692
Aligning gazeX data for 020: 2984 -> 1692
Aligning gazeY data for 020: 2984 -> 1692
Aligning gazeX data for 023: 2984 -> 1692
Aligning gazeY data for 023: 2984 -> 1692
Aligning gazeX data for 031: 2984 -> 1692
Aligning gazeY data for 031: 2984 -> 1692
Aligning gazeX data for 032: 2984 -> 1692
Aligning gazeY data for 032: 2984 -> 1692
Aligning gazeX data for 040: 2984 -> 1692
Aligning gazeY data for 040: 2984 -> 1692


Dimension,Participant_ID,valence,arousal,gazeX,gazeY
Participant_ID,,,,,
0,BF087_2,NaN,NaN,-0.527395,-0.860060
1,BF083_2,NaN,NaN,0.215383,-0.149729
2,BF090_1,NaN,NaN,-1.113760,0.398920
3,BF007_1,NaN,NaN,-1.788264,-0.493689
4,BF089_1,NaN,NaN,-1.028334,-0.354196
...,...,...,...,...,...
98,BF004_1,NaN,NaN,-0.073770,-0.311111
99,BF051_1,NaN,NaN,0.937927,0.015768
100,BF028_1,NaN,NaN,-0.828860,-0.478280


In [88]:
import pandas as pd
import numpy as np

def calculate_max_diff_error(df):
    """
    Calculates valence and arousal error columns by subtracting each rating 
    from the maximum observed finite rating in that column. Inf values are ignored 
    during the max calculation to prevent skewing.
    
    Args:
        df (pd.DataFrame): DataFrame containing 'valence' and 'arousal' columns.
        
    Returns:
        pd.DataFrame: DataFrame with 'valence_error' and 'arousal_error' columns added.
    """
    
    df = df.copy() 
    
    # --- Function to find max over finite values ---
    def safe_max(series):
        # Filter out NaN and Inf values
        finite_series = series[np.isfinite(series)]
        if finite_series.empty:
            return np.nan
        return finite_series.max()

    # 1. Calculate Maximums (ignoring NaNs and Infs)
    max_valence = safe_max(df['valence'])
    max_arousal = safe_max(df['arousal'])
    
    # 2. Calculate Error
    
    # Check if a valid max was found. If not, error should be NaN.
    if pd.isna(max_valence):
        df['valence_error'] = np.nan
    else:
        # Error = Max - Value. This calculation is vectorized.
        df['valence_error'] = max_valence - df['valence']

    if pd.isna(max_arousal):
        df['arousal_error'] = np.nan
    else:
        df['arousal_error'] = max_arousal - df['arousal']
    
    # 3. Calculate the combined error (average of the two errors)
    df['combined_error'] = (df['valence_error'] + df['arousal_error']) / 2
    
    return df

# Example Usage:
# data_df_with_errors = calculate_max_diff_error(data_df)
# results = lowess_gaze_optimization(data_df_with_errors, sigma=0.5)
# Example Usage:
# data_df_with_errors = calculate_max_diff_error(data_df)
# results = lowess_gaze_optimization(data_df_with_errors, sigma=0.5)
# Example Usage (replace data_df with your actual DataFrame name):
data_df_with_errors = calculate_max_diff_error(frame_indexed_data['011'][500])

In [89]:
data_df_with_errors

Dimension,Participant_ID,valence,arousal,gazeX,gazeY,valence_error,arousal_error,combined_error
Participant_ID,,,,,,,,
0,BF087_2,NaN,-inf,-0.117836,0.737671,NaN,inf,NaN
1,BF083_2,-0.629137,0.883670,-1.564539,-1.182665,1.615901,0.098789,0.857345
2,BF090_1,0.134687,0.143545,0.030341,0.071661,0.852077,0.838914,0.845495
3,BF007_1,0.715819,0.700960,-0.165491,0.450499,0.270944,0.281499,0.276221
4,BF089_1,0.958263,0.939686,-0.167139,0.444783,0.028500,0.042773,0.035637
...,...,...,...,...,...,...,...,...
98,BF004_1,0.821360,0.893787,-0.264484,1.158604,0.165403,0.088672,0.127038
99,BF051_1,-0.979630,0.526324,-0.484404,0.709403,1.966393,0.456135,1.211264
100,BF028_1,-0.575274,0.587889,-0.358639,-0.144329,1.562037,0.394570,0.978304


In [98]:
def lowess_gaze_optimization(data_df_with_errors, sigma=0.5):
    """
    Performs Kernel-Weighted Regression (LOWESS style) via grid search to 
    find the optimal gaze point (gazeX, gazeY) that minimizes the weighted 
    average of the combined error.

    Args:
        data_df_with_errors (pd.DataFrame): Data frame with 'gazeX', 'gazeY', 
                                            and 'combined_error' columns.
        sigma (float): The bandwidth parameter for the Gaussian kernel.

    Returns:
        dict: The optimal X, Y coordinates and the minimum error.
    """
    
    # --- Grid Definition (Adjusted for normalized coordinates -3 to 3) ---
    X_CANDIDATES = np.arange(-3.0, 3.0, 0.2) 
    Y_CANDIDATES = np.arange(-3.0, 3.0, 0.2) 
    
    # --- Prepare Data ---
    required_cols = ['gazeX', 'gazeY', 'combined_error']
    # Drop rows where we are missing any of the required values (gaze or error)
    data_points = data_df_with_errors.dropna(subset=required_cols).copy()
    
    if data_points.empty:
        return {'optimal_x': np.nan, 'optimal_y': np.nan, 'min_error': np.nan}

    all_candidate_errors = {}
    
    # --- Grid Search Loop ---
    for cx in X_CANDIDATES:
        for cy in Y_CANDIDATES:
            candidate_point = np.array([cx, cy])
            
            # Calculate the squared Euclidean distance from the candidate point 
            # to every actual gaze point.
            actual_gaze = data_points[['gazeX', 'gazeY']].values 
            squared_distance = np.sum((actual_gaze - candidate_point)**2, axis=1)
            
            # Calculate the Gaussian weights. 

            # Weight = exp(-(Distance^2) / (2 * sigma^2))
            weights = np.exp(-squared_distance / (2 * sigma**2))
            
            # Calculate Weighted Average Error
            weighted_error_sum = np.sum(weights * data_points['combined_error'].values)
            weight_sum = np.sum(weights)
            
            # Use epsilon to prevent division by zero (handles cases where sum of weights is zero)
            weighted_avg_error = weighted_error_sum / (weight_sum + 1e-9)
            
            all_candidate_errors[(cx, cy)] = weighted_avg_error

    # --- Find the Minimum Error ---
    optimal_point, min_error = min(all_candidate_errors.items(), key=lambda item: item[1])
    
    return {
        'optimal_x': optimal_point[0], 
        'optimal_y': optimal_point[1], 
        'min_error': min_error
    }

# Example Usage:
# 1. First, calculate the errors:
data_with_errors = calculate_max_diff_error(frame_indexed_data['011'][500])
# 
# 2. Then, run the optimization:
results = lowess_gaze_optimization(data_with_errors.dropna(), sigma=0.5)

/home/Jeff/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/Jeff/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/Jeff/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/Jeff/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/Jeff/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/Jeff/anaconda3/lib/python3.11/site-pack

In [99]:
results

{'optimal_x': -3.0, 'optimal_y': -3.0, 'min_error': nan}

In [83]:
video_id_to_process = '004'
# Create the expected structure: {'004': {frame_num: df, ...}}
input_data_for_function = {video_id_to_process: frame_indexed_data[video_id_to_process]}

optimal_gaze = find_optimal_gaze_point_FIXED(frame_indexed_data['011'][100])

AttributeError: 'str' object has no attribute 'dropna'

In [79]:
optimal_gaze

{'004': {'optimal_x': -3.0, 'optimal_y': -3.0, 'min_error': inf}}